In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DQN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(input_dim, 16)  # First hidden layer
        self.fc2 = nn.Linear(16, 16)         # Second hidden layer
        self.fc3 = nn.Linear(16, 16)         # Second hidden layer
        self.out = nn.Linear(16, output_dim) # Output layer (Q-values)

    def forward(self, x):
        x = F.relu(self.fc1(x))   # Activation after first hidden layer
        x = F.relu(self.fc2(x))   # Activation after second hidden layer
        x = F.relu(self.fc3(x))   # Activation after second hidden layer
        return self.out(x)        # Output raw Q-values (no activation)


In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import gym
import random
from collections import deque

class DQNAgent:
    def __init__(self, env, model, epsilon=0.1, gamma=0.99, lr=0.001, batch_size=64, memory_size=10000):
        self.env = env
        self.model = model
        self.target_model = DQN(env.observation_space.shape[0], env.action_space.n)
        self.target_model.load_state_dict(self.model.state_dict())  # Initialize target model with the same weights
        self.optimizer = optim.Adam(self.model.parameters(), lr=lr)
        self.criterion = nn.MSELoss()
        self.epsilon = epsilon
        self.gamma = gamma
        self.batch_size = batch_size
        self.memory = deque(maxlen=memory_size)
    
    def select_action(self, state):
        if random.random() < self.epsilon:  # Exploration: random action
            return self.env.action_space.sample()
        else:  # Exploitation: best action according to Q-network
            with torch.no_grad():
                state = torch.tensor(state, dtype=torch.float32).unsqueeze(0)  # Add batch dimension
                q_values = self.model(state)
                return torch.argmax(q_values, dim=1).item()

    def store_experience(self, experience):
        self.memory.append(experience)

    def sample_batch(self):
        batch = random.sample(self.memory, self.batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        states = torch.tensor(states, dtype=torch.float32)
        actions = torch.tensor(actions, dtype=torch.long)
        rewards = torch.tensor(rewards, dtype=torch.float32)
        next_states = torch.tensor(next_states, dtype=torch.float32)
        dones = torch.tensor(dones, dtype=torch.bool)
        return states, actions, rewards, next_states, dones

    def update_target_model(self):
        self.target_model.load_state_dict(self.model.state_dict())  # Copy weights from model to target model

    def train(self):
        if len(self.memory) < self.batch_size:
            return

        states, actions, rewards, next_states, dones = self.sample_batch()

        # Get Q-values for current states
        q_values = self.model(states)
        q_value = q_values.gather(1, actions.unsqueeze(1))

        # Get max Q-value for next states (target Q-value)
        with torch.no_grad():
            next_q_values = self.target_model(next_states)
            next_q_value = next_q_values.max(1)[0]
            target = rewards + (self.gamma * next_q_value * ~dones)

        # Compute loss and update weights
        loss = self.criterion(q_value.squeeze(1), target)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()


In [10]:
import gym

env = gym.make('CartPole-v1')
state_size = env.observation_space.shape[0]
action_size = env.action_space.n

In [19]:
def train_dqn(agent, episodes=1000):
    for episode in range(episodes):
        state = agent.env.reset()[0]  # Get the initial state
        done = False
        total_reward = 0

        while not done:
            action = agent.select_action(state)  # Choose an action
            next_state, reward, done, _, _ = agent.env.step(action)  # Take the action
            agent.store_experience((state, action, reward, next_state, done))  # Store experience

            state = next_state  # Move to next state
            total_reward += reward

            agent.train()  # Train on the stored experience

        # Update target model
        if episode % 10 == 0:
            agent.update_target_model()

        print(f"Episode {episode}, Total Reward: {total_reward}")

# # Create the environment and the agent
# env = gym.make('CartPole-v1')
# model = DQN(env.observation_space.shape[0], env.action_space.n)
# agent = DQNAgent(env, model)

# Train the agent
agent.target_model.train()
train_dqn(agent)


Episode 0, Total Reward: 125.0
Episode 1, Total Reward: 50.0
Episode 2, Total Reward: 119.0
Episode 3, Total Reward: 112.0
Episode 4, Total Reward: 117.0
Episode 5, Total Reward: 120.0
Episode 6, Total Reward: 118.0
Episode 7, Total Reward: 122.0
Episode 8, Total Reward: 118.0
Episode 9, Total Reward: 124.0
Episode 10, Total Reward: 120.0
Episode 11, Total Reward: 110.0
Episode 12, Total Reward: 121.0
Episode 13, Total Reward: 122.0
Episode 14, Total Reward: 107.0
Episode 15, Total Reward: 117.0
Episode 16, Total Reward: 115.0
Episode 17, Total Reward: 122.0
Episode 18, Total Reward: 116.0
Episode 19, Total Reward: 116.0
Episode 20, Total Reward: 117.0
Episode 21, Total Reward: 145.0
Episode 22, Total Reward: 137.0
Episode 23, Total Reward: 125.0
Episode 24, Total Reward: 161.0
Episode 25, Total Reward: 166.0
Episode 26, Total Reward: 126.0
Episode 27, Total Reward: 122.0
Episode 28, Total Reward: 48.0
Episode 29, Total Reward: 139.0
Episode 30, Total Reward: 130.0
Episode 31, Total Re

In [ ]:
def test_dqn(agent, episodes=1):
    total_rewards = []
    for episode in range(episodes):
        state = agent.env.reset()[0]  # Get the initial state
        done = False
        total_reward = 0

        while not done:
            # Select action based on the current policy (greedy, no exploration)
            action = agent.select_action(state)  # Select action (epsilon=0, so purely greedy)
            next_state, reward, done, _, _ = agent.env.step(action)  # Take action
            state = next_state  # Move to next state
            total_reward += reward  # Accumulate total reward

        total_rewards.append(total_reward)
        print(f"Test Episode {episode + 1}, Total Reward: {total_reward}")

    avg_reward = np.mean(total_rewards)
    print(f"Average Test Reward over {episodes} episodes: {avg_reward}")
    return avg_reward

# Put the target model in eval mode before testing
agent.target_model.eval()

# Run the test loop
test_dqn(agent)


(4,)


KeyboardInterrupt: 